# JEPA For Time Series

> I guess so!

In [ ]:
#| default_exp jepa

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, math, torch.nn.functional as F, torch.nn as nn, copy, numpy as np, lightning.pytorch as pl, warnings

from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingWarmRestarts
from physiojepa.layers import MultiHeadAttention, MLP, Patch, tAPE, PositionalEncoding
from physiojepa.tokenizers import TS_Tokenizer, TS_Tokenizer_Complex, InceptionTokenizer, PatchEncoder
from physiojepa.utils import trunc_normal_
from physiojepa.patchtst import TSTBlock

## Torch

In [ ]:
#| export
class JEPABlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        mlp_ratio=4.0,
        qkv_bias=False,
        qk_scale=None,
        drop=0.,
        attn_drop=0.,
        act_layer=nn.GELU,
        norm_layer=nn.LayerNorm,
        rotary_pes=False
    ):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = MultiHeadAttention(
            dim,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            qk_scale=qk_scale,
            attn_drop=attn_drop,
            proj_drop=drop,
            rotary_pes=rotary_pes
            )

        self.norm2 = norm_layer(dim)
        self.rotary_pes = rotary_pes 
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = MLP(
            in_features=dim,
            hidden_features=mlp_hidden_dim,
            act_layer=act_layer,
            drop=drop)

    def forward(self, x, mask=None):
        x = self.norm1(x)
        y = self.attn(x, key=x, value=x, mask=mask)
        x = x + y
        x = x + self.mlp(self.norm2(x))
        return x

def apply_masks(x, masks):
    all_x = []
    for m, x_i in zip(masks, x):
        mask_keep = m.unsqueeze(-1).repeat(1, x_i.size(-1))
        x_i_masked = torch.gather(x_i, dim=0, index=mask_keep)
        assert x_i_masked.shape[0] == mask_keep.shape[0], "The number of masked tokens does not match the number of masked positions"
        all_x.append(x_i_masked)
        return torch.stack(all_x, dim=0)


def create_masks(x, patch_size, patch_stride, context_mask_range, target_mask_range, melt_channels_to_batch=False):
    # Create the mask for the patches

    batch_size = 0
    channel_size = x.size(1)
    sample_n_patches = []
    for x_i in x:
        num_patches = int((max(x_i.size(-1), patch_size)-patch_size) // patch_stride + 1)
        if ((x_i.size(-1)-patch_size) % patch_stride != 0):
            num_patches += 1
        if melt_channels_to_batch:
            added_batches = channel_size
            batch_size += added_batches
            sample_n_patches.extend(num_patches for i in range(added_batches))
        else:
            batch_size += 1
            sample_n_patches.append(num_patches)
    
    target_indices_t = []
    context_indices_t = []

    for i, n_patch in enumerate(sample_n_patches):
        target_ratio = target_mask_range[0] + torch.rand(1).item() * (target_mask_range[1] - target_mask_range[0])
        context_ratio = context_mask_range[0] + torch.rand(1).item() * (context_mask_range[1] - context_mask_range[0])
        perm = torch.randperm(n_patch)
        num_target = int(n_patch * target_ratio)
        remaining = n_patch - num_target
        num_context = int(remaining * context_ratio)
                    
        target_indices = perm[:num_target]
        context_indices = perm[num_target:num_target + num_context]

        assert len(target_indices) > 0, "No target indices created"
        assert len(context_indices) > 0, "No context indices created"
        assert set(target_indices).isdisjoint(context_indices), "Target and context indices overlap"

        target_indices_t.append(target_indices)
        context_indices_t.append(context_indices)
    min_target_len = min(len(mask) for mask in target_indices_t) # make them all the same
    min_context_len = min(len(mask) for mask in context_indices_t)

    mask_indices = torch.zeros((batch_size, min_target_len), dtype=torch.long)
    non_mask_indices = torch.zeros((batch_size, min_context_len), dtype=torch.long)

    for i in range(batch_size):
        mask_indices[i, :] = target_indices_t[i][:min_target_len]
        non_mask_indices[i, :] = context_indices_t[i][:min_context_len]
    return mask_indices, non_mask_indices

In [ ]:
#| export
class Encoder(nn.Module):
    def __init__(
        self,
        c_in,
        num_patches,
        patch_size,
        patch_stride,
        d_model,
        nhead,
        num_layers,
        use_tst_block=False,
        shared_embedding=True,
        pe_type='tAPE',
        mlp_ratio=4.0,
        qkv_bias=True,
        qk_scale=None,
        drop_rate=0.0,
        attn_drop_rate=0.0,
        norm_layer=nn.LayerNorm,
        jepa=True,
        embed_activation=nn.GELU(),
        init_std=0.02,
        tokenizer_type='simple',
        tokenizer_kwargs={}
    ):

        super().__init__()

        # Parameters
        self.c_in = c_in
        self.patch_size = patch_size
        self.patch_stride = patch_stride
        self.d_model = d_model
        self.num_patches = num_patches
        self.activation = embed_activation if embed_activation else nn.GELU()
        self.init_std = init_std
        self.pe_type = pe_type.lower()
        self.tokenizer_type = tokenizer_type.lower()
        self.shared_embedding = shared_embedding
        self.use_tst_block = use_tst_block

        self.patch_layer = Patch(patch_len=patch_size, stride=patch_stride)
        # Building the tokenizer
        if self.tokenizer_type in ['simple_conv', 'simple']:
            self.tokenizer = TS_Tokenizer(
                c_in=c_in,
                patch_size=patch_size,
                d_model=d_model * c_in if not shared_embedding else d_model,
                patch_stride=patch_stride,
                shared_embedding=self.shared_embedding
            )
        elif self.tokenizer_type in ['complex_conv', 'complex']:
            self.tokenizer = TS_Tokenizer_Complex(
                c_in=c_in,
                patch_size=patch_size,
                d_model=d_model
            )
        elif self.tokenizer_type == 'linear':
            self.tokenizer = PatchEncoder(c_in=c_in, 
                                             patch_len=patch_size, 
                                             d_model=d_model,
                                             shared_embedding=self.shared_embedding
                                             )
        elif self.tokenizer_type == 'inception':
            self.tokenizer = InceptionTokenizer(c_in=c_in, 
                                                  patch_size=patch_size,
                                                  d_model=d_model * c_in if not shared_embedding else d_model,
                                                  patch_stride=patch_stride,
                                                  shared_embedding=self.shared_embedding,
                                                  **tokenizer_kwargs
                                                  )
        else:
            raise ValueError(f"Invalid tokenizer type: {tokenizer_type}. Valid options are: 'simple_conv', 'complex_conv', 'linear'")

        # Positional Encoder -- We use a Sin-Cos one (not learnable)
        if self.pe_type == 'tape':
            self.pe = tAPE(d_model=self.d_model, seq_len=self.num_patches)
        elif self.pe_type == 'learned':
            self.pe = PositionalEncoding(num_patch=self.num_patches, d_model=self.d_model)
        elif self.pe_type == 'rotary':
            self.pe = nn.Identity()
        
        if self.use_tst_block:
            self.dropout = nn.Dropout(drop_rate) # residual dropout
        else:
            self.dropout = nn.Identity()

        # Transformer part of the encoder
        if not use_tst_block:
            self.predictor_blocks = nn.ModuleList(
                [
                    JEPABlock(
                        dim=self.d_model,
                        num_heads=nhead,
                        mlp_ratio=mlp_ratio,
                        qkv_bias=qkv_bias,
                        qk_scale=qk_scale,
                        drop=drop_rate,
                        attn_drop=attn_drop_rate,
                        act_layer=nn.GELU,
                        norm_layer=norm_layer,
                        rotary_pes=self.pe_type == 'rotary'
                    )
                    for i in range(num_layers)
                ]
            )
        else:
            self.predictor_blocks = nn.ModuleList([TSTBlock(d_model=self.d_model, 
                                                n_heads=nhead, 
                                                d_ff=int(self.d_model * mlp_ratio), 
                                                attn_dropout=attn_drop_rate, 
                                                dropout=drop_rate, 
                                                bias=qkv_bias,
                                                activation='gelu', 
                                                pre_norm=False, 
                                                rotary_pes=self.pe_type == 'rotary') for _ in range(num_layers)])

        if not use_tst_block:
            self.encoder_norm = nn.LayerNorm(self.d_model)
        else:
            self.encoder_norm = nn.Identity()
        self.jepa = jepa
        self.apply(self._init_weights)
        self._rescale_blocks()

    def forward(self, x, mask=None):
        # Embed the data using the Tokenizer
        bs = x.size(0)

        if self.tokenizer_type == 'linear':
            x = self.patch_layer(x, constant_pad=True, constant_pad_value=0) 
        x = self.tokenizer(x) # [bs x num_patches x (?c_in) x d_model]

        if x.dim() == 3:
            x = x.unsqueeze(2) # z: [bs x num_patch x 1 x d_model]
        x = x.transpose(1,2)
        transformer_c_in = x.size(1)
        x = torch.reshape(x, (bs * transformer_c_in, self.num_patches, self.d_model)) # u: [bs * nvars x num_patch x d_model]
        #batch_size, num_patches, _ = x.size()
        x = self.pe(x)
        x = self.dropout(x)
        # Apply mask -- In the encoder, we keep only the unmasked part
        if mask is not None and self.jepa:
            x = apply_masks(x, mask)
        # Encode using Attention
        for blk in self.predictor_blocks:
            x = blk(x, mask=None)
       
        x = self.encoder_norm(x)
        return x
    
    def _rescale_blocks(self):
        def rescale(param, layer_id):
            param.div_(math.sqrt(2.0 * layer_id))
        if not self.use_tst_block:
            for layer_id, layer in enumerate(self.predictor_blocks):
                rescale(layer.attn.proj.weight.data, layer_id + 1) # rescale the attention weights
                rescale(layer.mlp.fc2.weight.data, layer_id + 1) # rescale the feedforward weights
        else:
            for layer_id, layer in enumerate(self.predictor_blocks):
                rescale(layer.self_attn.proj.weight.data, layer_id + 1) # rescale the attention weights
                rescale(layer.ff[3].weight.data, layer_id + 1) # rescale the feedforward weights

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv1d) or isinstance(m, nn.Conv2d):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

In [ ]:
#| export
class Predictor(nn.Module):
    def __init__(
        self,
        num_patches,
        encoder_embed_dim=128,
        predictor_embed_dim=128,
        nhead=2,
        num_layers=1,
        use_tst_block=False,
        pe_type='tAPE',
        mlp_ratio=4.0,
        qkv_bias=True,
        qk_scale=None,
        drop_rate=0.0,
        attn_drop_rate=0.0,
        norm_layer=nn.LayerNorm,
        embed_activation=nn.GELU(),
        init_std=0.02,
        c_in_mask_tokens = 1, # number of channels in the encoder (if treating channels sep)
    ):
        super(Predictor, self).__init__()

        # Model's parameters
        self.activation = embed_activation if embed_activation else nn.GELU()
        self.predictor_embed_dim = predictor_embed_dim
        self.num_patches = num_patches
        self.init_std = init_std
        self.pe_type = pe_type.lower()
        self.use_tst_block = use_tst_block
        self.c_in_mask_tokens = c_in_mask_tokens
        # Map the Encoder's embed dim to the predictor's embed dim
        self.predictor_embed = nn.Linear(
            encoder_embed_dim, predictor_embed_dim, bias=True
        )

        # Positional Encoder -- Note that we are using a Sin-Cos PE
        if self.pe_type == 'tape':
            self.pos_embed = nn.Parameter(
               torch.zeros(1, self.num_patches, self.predictor_embed_dim), requires_grad=False
            )
            self.init_tape_pe()
        elif self.pe_type == 'learned':
            # interpolated PEs
            n_learned_pes = min(2048, self.num_patches//4)
            self.pos_embed =  nn.Parameter(torch.empty((n_learned_pes, predictor_embed_dim)))
            nn.init.uniform_(self.pos_embed, -0.02, 0.02)
        elif self.pe_type in ['rotary', 'none']:
            self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, self.predictor_embed_dim), requires_grad=False)
        else:
            self.pos_embed = nn.Parameter(
               torch.zeros(1, self.num_patches, self.predictor_embed_dim), requires_grad=False
            )
            self.init_embed()

        if use_tst_block:
            self.dropout = nn.Dropout(drop_rate) # residual dropout
        else:
            self.dropout = nn.Identity()

        # Mask tokens
        self.mask_token = nn.Parameter(torch.zeros(c_in_mask_tokens, 1, predictor_embed_dim), requires_grad=True)
        self.mask_token = trunc_normal_(self.mask_token, std=init_std)

        # Transformer part of the Decoder
        if not use_tst_block:
            self.predictor_blocks = nn.ModuleList(
                [
                    JEPABlock(
                        dim=predictor_embed_dim,
                        num_heads=nhead,
                        mlp_ratio=mlp_ratio,
                        qkv_bias=qkv_bias,
                        qk_scale=qk_scale,
                        drop=drop_rate,
                        attn_drop=attn_drop_rate,
                        act_layer=nn.GELU,
                        norm_layer=norm_layer,
                        rotary_pes = self.pe_type == 'rotary'
                    )
                    for i in range(num_layers)
                ]
            )
        else:
            self.predictor_blocks = nn.ModuleList([TSTBlock(d_model=predictor_embed_dim, 
                                                n_heads=nhead, 
                                                d_ff=int(predictor_embed_dim * mlp_ratio),
                                                attn_dropout=attn_drop_rate, 
                                                dropout=drop_rate, 
                                                bias=qkv_bias,
                                                activation='gelu', 
                                                pre_norm=False, 
                                                rotary_pes=self.pe_type == 'rotary') for _ in range(num_layers)])

        # To Normalize and map back to the encoder dimension (before applying
        # the loss function)
        if not use_tst_block:
            self.predictor_norm = nn.LayerNorm(predictor_embed_dim)
        else:
            self.predictor_norm = nn.Identity()
        self.predictor_proj = nn.Linear(
            predictor_embed_dim, encoder_embed_dim, bias=True
        )

        self.apply(self._init_weights)
        self._rescale_blocks()

    def forward(self, encoded_vals, mask=None, non_masks=None):

        assert (mask is not None) and (encoded_vals is not None), "No input found"
        
        _, ctx_size, _ = encoded_vals.size()
        batch_size = encoded_vals.size(0)

        # Map the output of the encoder to the Predictor's dimension
        x = self.predictor_embed(encoded_vals)

        if self.pe_type == 'learned':
            # interpolate
            pos_embed = self.pos_embed.unsqueeze(0).permute(0,2,1) # [1,d,N]
            pos_embed = F.interpolate(pos_embed, size=self.num_patches, mode='linear', align_corners=False)
            pos_embed = pos_embed.permute(0,2,1) # 1, N , d
        else:
            pos_embed = self.pos_embed

        # Add PE and apply mask to keep only the non-masked part
        cnt_pos_enc = pos_embed.repeat(batch_size, 1, 1)
        
        cnt_pos_enc = apply_masks(cnt_pos_enc, non_masks)
        x = x + cnt_pos_enc
        x = self.dropout(x)

        # Create the Target vectors and add PE
        target_pos_enc = pos_embed.repeat(batch_size, 1, 1)
        
        target_pos_enc = apply_masks(target_pos_enc, mask)
        pred_tokens = self.mask_token.repeat(batch_size // self.c_in_mask_tokens, target_pos_enc.size(1), 1)
        pred_tokens = pred_tokens + target_pos_enc
        pred_tokens = self.dropout(pred_tokens)
        # Concat the context (from the encoder) and the mask tokens
        
        x = torch.cat([x, pred_tokens], dim=1)
        shuffled_ = torch.randperm(x.shape[1]) # random permutation of patch indices
        ids_restore = torch.argsort(shuffled_) # restore indices
        x = x [:,shuffled_,:] # shuffle x
        # Push through attention
        for blk in self.predictor_blocks:
            x = blk(x, mask=None)

        x = self.predictor_norm(x)

        # Output only the part related to the masked area and adapt the dim
        x = x[:, ids_restore] # restore order
        x = x[:, ctx_size:] # grab targets
        assert mask.shape[1] == x.shape[1], "The target mask shape does not equal the final shape"
        x = self.predictor_proj(x)

        return x
    
    def _rescale_blocks(self):
        def rescale(param, layer_id):
            param.div_(math.sqrt(2.0 * layer_id))
        if not self.use_tst_block:
            for layer_id, layer in enumerate(self.predictor_blocks):
                rescale(layer.attn.proj.weight.data, layer_id + 1) # rescale the attention weights
                rescale(layer.mlp.fc2.weight.data, layer_id + 1) # rescale the feedforward weights
        else:
            for layer_id, layer in enumerate(self.predictor_blocks):
                rescale(layer.self_attn.proj.weight.data, layer_id + 1) # rescale the attention weights
                rescale(layer.ff[3].weight.data, layer_id + 1) # rescale the feedforward weights

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0) 

    def init_embed(self):
        """
        This function serves for the positional encoder which is based on a
        Sin-Cos Pos Encoder.
        ---
        Users can choose any other Positional Encoder that they may see fit.
        """
        assert self.predictor_embed_dim % 2 == 0

        omega = np.arange(self.predictor_embed_dim // 2, dtype=float)
        omega /= self.predictor_embed_dim / 2.0
        omega = 1.0 / 10000**omega

        pos = np.arange(self.num_patches, dtype=float)
        pos = pos.reshape(-1)
        out = np.einsum("m,d->md", pos, omega)

        emb_sin = np.sin(out)
        emb_cos = np.cos(out)

        emb = np.concatenate([emb_sin, emb_cos], axis=1)

        self.pos_embed.data.copy_(torch.from_numpy(emb).float().unsqueeze(0))

        return emb
    
    def init_tape_pe(self):
        """
        This function serves for the positional encoder which is based on a
        Sin-Cos Pos Encoder.
        ---
        Users can choose any other Positional Encoder that they may see fit.
        """
        assert self.predictor_embed_dim % 2 == 0
        pos = torch.arange(0, self.num_patches, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, self.predictor_embed_dim, 2).float() * (-np.log(10000.0) / self.predictor_embed_dim))
        W_pos = torch.zeros(self.num_patches, self.predictor_embed_dim)
        W_pos[:, 0::2] = torch.sin((pos * div_term)*(self.predictor_embed_dim/self.num_patches)) # this is the difference between normal PE and tAPE, scaling (d_model/seq_len)
        W_pos[:, 1::2] = torch.cos((pos * div_term)*(self.predictor_embed_dim/self.num_patches))

        self.pos_embed.data.copy_(W_pos.unsqueeze(0))

        return W_pos
   
def variance_loss(x):
    return torch.mean(F.relu(1.0 - x.std(dim=1)).mean())

def loss_pred(pred, target_ema, representations=None, alpha = 0.2):
    loss = 0.0
    for pred_i, target_ema_i in zip(pred, target_ema):
        loss = loss + F.mse_loss(pred_i, target_ema_i, reduction='mean')
    loss /= len(pred)
    return loss

def mse_variance_loss(pred, target_ema, representations, alpha = 0.2):
    loss = 0.0
    for pred_i, target_ema_i, representations_i in zip(pred, target_ema, representations):
        loss = loss + F.mse_loss(pred_i, target_ema_i, reduction='mean')
        loss = loss + alpha * F.relu(1.0 - representations_i.std(dim=-1)).mean()
    loss /= len(pred)
    return loss

In [ ]:
#| export
class JEPASimpleLightning(pl.LightningModule):
    def __init__(self,
                 learning_rate,
                 train_size,
                 batch_size,
                 n_gpus,
                 patchtsjepa_encoder_kwargs,
                 patchtsjepa_predictor_kwargs,
                 weight_decay=0.04,
                 use_weight_decay_scheduler=False,
                 final_weight_decay=0.4,
                 epochs=100,
                 optimizer_type='adamw',
                 scheduler_type='OneCycle',
                 target_mask_range=(0.05,0.3), # the target can be up to 50% of the original x 
                 context_mask_range=(0.5, 1.), # the context can be up to 80% of masked out target (1-target_mask_ratio)
                 mask_block_range=(1, 30),
                 ema_decay=0.996,
                 scheduler_kwargs={},
                 transforms=None,
                 loss_fn=loss_pred
                 ):
        super().__init__()
        self.scheduler_type = scheduler_type.lower()
        self.scheduler_kwargs = scheduler_kwargs
        if self.scheduler_type is not None:
            assert self.scheduler_type in ['onecycle', 'cosineannealingwarmrestarts'], "scheduler must be either OneCycle, CosineAnnealingWarmRestarts, or None"
        self.save_hyperparameters()
        self.learning_rate = learning_rate
        self.train_size = train_size
        self.batch_size = batch_size * n_gpus
        self.epochs = epochs
        self.optimizer_type = optimizer_type.lower()
        self.weight_decay = weight_decay
        self.use_weight_decay_scheduler = use_weight_decay_scheduler
        self.final_weight_decay = final_weight_decay
        self.loss_fn = loss_fn
        self.target_mask_range = target_mask_range
        self.context_mask_range = context_mask_range
        self.ema_decay = ema_decay
        self.ipe = self.train_size//self.batch_size
        self.total_steps = int(self.ipe*self.epochs)
        self.encoder = Encoder(**patchtsjepa_encoder_kwargs)
        self.num_patch = self.encoder.num_patches
        self.patch_size = self.encoder.patch_size
        self.patch_stride = self.encoder.patch_stride
        self.mask_block_range = mask_block_range
        self.c_in = self.encoder.c_in
        self.d_model = self.encoder.d_model
        patchtsjepa_predictor_kwargs['num_patches'] = self.num_patch
        self.predictor = Predictor(**patchtsjepa_predictor_kwargs)
        self.target_encoder = copy.deepcopy(self.encoder).requires_grad_(False) # no grad, ema weight updates
        self.transforms = transforms
        self.tokenizer_type = patchtsjepa_encoder_kwargs.get('tokenizer_type')
        self.melt_channels_to_batch = (not patchtsjepa_encoder_kwargs.get('shared_embedding') and self.tokenizer_type in ['inception', 'simple', 'simple_conv']) or (self.tokenizer_type == 'linear')
        
    def get_momentum_value(self):
        """Calculate momentum value based on current training step"""
        current_step = self.global_step  # PyTorch Lightning tracks this automatically
        # Ensure we don't exceed maximum momentum
        progress = min(current_step / self.total_steps, 1.0)
        # Linear warmup from ema_decay to 1.0
        momentum = self.ema_decay + progress * (1.0 - self.ema_decay)
        return momentum
    

    def ema_update(self, context_encoder, target_encoder):
        with torch.no_grad():
            m = self.get_momentum_value()
            for param_q, param_k in zip(context_encoder.parameters(), target_encoder.parameters()):
                param_k.data.mul_(m).add_((1.-m) * param_q.detach().data)
                param_k.requires_grad_(False)
        return target_encoder
    
    def forward(self, x, channel_mask=None):
        """
        should output: [bs x nvars x d_model x num_patch] 
        """
        # use context for forward
        x = self.encoder(x) # bs x n_patch x d_model
        if self.melt_channels_to_batch:
            bs = x.size(0) // self.c_in
            x = x.reshape(bs, self.c_in, -1, self.d_model)
        else:
            x = x.unsqueeze(1) # add back channel dim
        x = x.permute(0,1,3,2) # bs x nvars x d_model x n_patch, 
        return x

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        if self.transforms is not None:
            batch = self.transforms(batch)
        x, _ = batch
        bs = x.size(0)
        
        masks, non_masks = create_masks(x, patch_size=self.patch_size, patch_stride=self.patch_stride, context_mask_range=self.context_mask_range, target_mask_range=self.target_mask_range, melt_channels_to_batch=self.melt_channels_to_batch)
        masks = masks.to(x.device)
        non_masks = non_masks.to(x.device)
          
        # Predict targets
        with torch.no_grad():
            target_ema = self.target_encoder(x)
            target_ema = F.layer_norm(
                target_ema, (target_ema.size(-1),)
            )  # normalize over feature-dim  [B, N, D]
            target_ema = apply_masks(target_ema, masks)
        
        # Encode and Predict the masked tokens
        tokens = self.encoder(x, mask=non_masks)

        pred = self.predictor(tokens, mask=masks, non_masks=non_masks)

        # Compute the loss
        loss = self.loss_fn(pred, target_ema, representations=tokens, alpha=0.2)

        if loss.isnan():
            warnings.warn(f"""
            Train Loss is NaN at batch idx {batch_idx}
            """)
            loss = torch.nan_to_num(loss) # convert to 0
        
        if batch_idx % 50 == 0:
            with torch.no_grad():
                # Check representation stats
                tokens = tokens.clone().detach()
                target_ema = target_ema.clone().detach()
                pred = pred.clone().detach()

                context_mean = tokens.mean().item()
                target_mean = target_ema.mean().item()
                target_std = target_ema.std().item()
                pred_mean = pred.mean().item()
                pred_std = pred.std().item()

                num_target_masks = torch.numel(masks)
                num_context_masks = torch.numel(non_masks)

                tokens = torch.reshape(tokens, (bs, self.c_in if self.melt_channels_to_batch else 1, -1, self.d_model)) # z: [bs x nvars x num_patch x d_model]
                target_ema = torch.reshape(target_ema, (bs, self.c_in if self.melt_channels_to_batch else 1, -1, self.d_model)) # z: [bs x nvars x num_patch x d_model]
                pred = torch.reshape(pred, (bs, self.c_in if self.melt_channels_to_batch else 1, -1, self.d_model)) # z: [bs x nvars x num_patch x d_model]

                bs, C, num_patches, d_model = pred.shape
                sample_size = min(1000, num_patches)
                indices = torch.randperm(num_patches)[:sample_size] # random samples
                pred = pred.permute(0,2,1,3) # bs x num_patches x c_in x d_model
                pred = pred[:, indices, :, :]
                flat = pred.view(bs * sample_size, C, d_model)
                
                for i in range(C):
                    norm = F.normalize(flat[:,i,:], dim=1)
                    similarity_matrix = norm @ norm.T
                    mean = pred[:,i,:].mean().item()
                    std = pred[:,i,:].std().item()
                    pred_pairwise_cos_sim = similarity_matrix[~torch.eye(bs*sample_size, dtype=torch.bool)].mean().item()
                    self.log(f'pred_cos_sim_channel_{i}', pred_pairwise_cos_sim)
                    self.log(f'pred_mean_channel_{i}', mean)
                    self.log(f'pred_std_channel_{i}', std)

                bs, C, num_patches, d_model = tokens.shape
                sample_size = min(1000, num_patches)
                indices = torch.randperm(num_patches)[:sample_size] # random samples
                tokens = tokens.permute(0,2,1,3) # bs x num_patches x c_in x d_model
                tokens = tokens[:, indices, :, :]
                flat = tokens.view(bs * sample_size, C, d_model)
                
                for i in range(C):
                    norm = F.normalize(flat[:,i,:], dim=1)
                    similarity_matrix = norm @ norm.T
                    mean = tokens[:,i,:].mean().item()
                    std = tokens[:,i,:].std().item()
                    context_pairwise_cos_sim = similarity_matrix[~torch.eye(bs*sample_size, dtype=torch.bool)].mean().item()
                    self.log(f'context_cos_sim_channel_{i}', context_pairwise_cos_sim)
                    self.log(f'context_mean_channel_{i}', mean)
                    self.log(f'context_std_channel_{i}', std)

                bs, C, num_patches, d_model = target_ema.shape
                sample_size = min(1000, num_patches)
                indices = torch.randperm(num_patches)[:sample_size] # random samples
                target_ema = target_ema.permute(0,2,1,3) # bs x num_patches x c_in x d_model
                target_ema = target_ema[:, indices, :, :]
                flat = target_ema.view(bs * sample_size, C, d_model)
                for i in range(C):
                    norm = F.normalize(flat[:,i,:], dim=1)
                    similarity_matrix = norm @ norm.T
                    mean = target_ema[:,i,:].mean().item()
                    std = target_ema[:,i,:].std().item()
                    target_pairwise_cos_sim = similarity_matrix[~torch.eye(bs*sample_size, dtype=torch.bool)].mean().item()
                    self.log(f'target_cos_sim_channel_{i}', target_pairwise_cos_sim)
                    self.log(f'target_mean_channel_{i}', mean)
                    self.log(f'target_std_channel_{i}', std)

                self.log('context_mean', context_mean)
                self.log('target_mean', target_mean)
                self.log('target_std', target_std)
                self.log('pred_mean', pred_mean)
                self.log('pred_std', pred_std)
                self.log('num_target_masks', num_target_masks)
                self.log('num_context_masks', num_context_masks)

        loss = loss.to(self.device)
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        return loss
    
    def on_train_batch_end(self, outputs, batch, batch_idx):
        """Called after each training batch ends"""
        # Update target encoder weights using EMA
        self.target_encoder = self.ema_update(
            self.encoder, 
            self.target_encoder
        )

    def on_train_batch_start(self, batch, batch_idx):
        # update weight decay
        if self.use_weight_decay_scheduler:
            step = self.global_step
            T_max = int(self.ipe * self.epochs)
            progress = step / T_max
            new_wd = self.final_weight_decay + (self.weight_decay - self.final_weight_decay) * 0.5 * (1. + math.cos(math.pi * progress))

            if self.final_weight_decay <= self.weight_decay:
                new_wd = max(self.final_weight_decay, new_wd)
            else:
                new_wd = min(self.final_weight_decay, new_wd)

            for group in self.optimizer.param_groups:
                if ('WD_exclude' not in group) or not group['WD_exclude']:
                    group['weight_decay'] = new_wd
    
    def validation_step(self, batch, batch_idx):
        x, _ = batch
        masks, non_masks = create_masks(x, patch_size=self.patch_size, patch_stride=self.patch_stride, context_mask_range=self.context_mask_range, target_mask_range=self.target_mask_range, melt_channels_to_batch=self.melt_channels_to_batch)
        masks = masks.to(x.device)
        non_masks = non_masks.to(x.device)
          
        # Predict targets
        with torch.no_grad():
            target_ema = self.target_encoder(x)
            target_ema = F.layer_norm(
                target_ema, (target_ema.size(-1),)
            )  # normalize over feature-dim  [B, N, D]
            target_ema = apply_masks(target_ema, masks)
        
        # Encode and Predict the masked tokens
        tokens = self.encoder(x, mask=non_masks)

        pred = self.predictor(tokens, mask=masks, non_masks=non_masks)

        # Compute the loss
        loss = self.loss_fn(pred, target_ema, representations=tokens, alpha=0.2)

        if loss.isnan():
            warnings.warn(f"""
            Val Loss is NaN at batch idx {batch_idx}
            """)
            loss = torch.nan_to_num(loss) # convert to 0

        loss = loss.to(self.device)

        self.log("val_loss", loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)

    def configure_optimizers(self):
        param_groups = [ # exclude bias and layer norm parameters from weight decay
            {
                'params': (p for n, p in self.encoder.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))
            }, {
                'params': (p for n, p in self.predictor.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))
            }, {
                'params': (p for n, p in self.encoder.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
                'WD_exclude': True,
                'weight_decay': 0,
            }, {
                'params': (p for n, p in self.predictor.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
                'WD_exclude': True,
                'weight_decay': 0,
            },
        ]
        self.optimizer = torch.optim.AdamW(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0.0, fused=False) if self.optimizer_type == 'adamw' else\
                     torch.optim.Adam(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0.0, fused=False)
        if self.scheduler_type == 'onecycle':
            scheduler = OneCycleLR(self.optimizer, epochs=self.epochs, steps_per_epoch=self.ipe, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(self.optimizer, **self.scheduler_kwargs) # T_0=self.epochs//10, T_mult=2, eta_min=1e-8) # lr max is initial LR
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return self.optimizer

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()